In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
google_api_key = os.getenv("GOOGLE_API_KEY")

In [19]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool
from langchain.agents import create_agent


In [4]:
loader =PyPDFLoader("../data/dermAI.pdf")
docs = loader.load()
len(docs)

32

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

34

In [6]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

In [7]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [8]:
## agents, llm, tools


In [24]:
@tool   
def retriever_tool(query:str):
    """
    This tool can help you to retrieve the relevant data from PDF Document and these documents have details about dermAI Project.
    """
    print("tool called", query)
    docs= vector_store.similarity_search(query=query, k=4)
    context =""
    for doc in docs:
        context= doc.page_content + "\n\n"

    return context

In [25]:
retriever_tool.invoke("dermAI")

tool called dermAI


'Depending on the selected dataset, categories may include:\nMelanoma\nMelanocytic nevus\nBasal cell carcinoma\nActinic keratosis\nBenign keratosis\nDermatofibroma\nVascular lesion\nOther dataset-defined skin conditions\nThe actual disease classes must be determined by the selected dataset and its\nground-truth labels.\nClassification Pipeline\nSkin Image →  Preprocessing →  Lesion ROI →  Feature Extraction →  Deep\nLearning Model →  Disease Prediction →  Confidence Score\n7. Advanced Deep Learning Intelligence Engine\nThis module provides the advanced AI capabilities of DermAI.\nCNN-Based Models\nResNet\nDenseNet\nEfficientNet\nMobileNet\nConvNeXt\nTransformer-Based Models\nVision Transformer\nSwin Transformer\nDermAI: Skin Disease Detector from Mobile Images Using DL\n9\n\n'

In [18]:
llm = ChatGroq(model="openai/gpt-oss-120b")


In [20]:
System_Prompt =""" 
    You are a helpful assistant that answers question using retrieval context.
    always use the retriever_tool or questions using external knowlwdge.

"""

In [21]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt

)

In [22]:
query="what is dermAI and what is the purpose of it?"
response = agent.invoke({"messages": [{"role":"user","content":query}]})

In [23]:
result = response["messages"][-1].content
print(result)


**DermAI** is a deep‑learning‑based application that analyzes photos of skin taken with a mobile device and predicts the likely skin condition.  

**Purpose**

- **Early detection & triage** – give users (patients, clinicians, or health‑care administrators) an instant, on‑the‑spot assessment of skin lesions so that suspicious cases can be referred for professional care sooner.  
- **Decision support** – provide clinicians with confidence scores, explainable‑AI visualizations (e.g., heat‑maps showing which image regions drove the prediction), and a history of previous predictions for each patient, helping them confirm or rule‑out a diagnosis.  
- **Workflow & reporting** – offer dashboards for different stakeholder groups, automated analysis reports, and case‑management tools that streamline follow‑up and monitoring.  
- **Scalability & reproducibility** – the whole system is containerized with Docker, making it easy to deploy in clinics, research settings, or cloud environments while a